In [1]:
## READS BBO CSV exported from SC raw .scid data, saves to parquet per contract

In [2]:
import pandas as pd
import numpy as np
import datetime as dt 
from pathlib import Path
import time

In [3]:
# Configuration: Style preferences
#plt.style.use('ggplot') # Good default for readability
pd.set_option("display.width", 400)      # total characters per line
pd.set_option("display.max_columns", 30) # prevent wrapping by limiting columns
pd.set_option("display.max_rows", 1000)

In [4]:
import os
os.getcwd()

'/home/vm/pt/hgt-rl/labeller'

In [5]:
%%time
symbol = 'mnq'
SEC = 2

inFile = f'/mnt/c/ML/DATA/2025/holy-grail/{symbol}-ha-{SEC}sec-labeller.csv'
outFile= f'data/{symbol}-ha-{SEC}sec-labeller.pqt'

print(inFile, outFile,)

/mnt/c/ML/DATA/2025/holy-grail/mnq-ha-2sec-labeller.csv data/mnq-ha-2sec-labeller.pqt
CPU times: user 292 μs, sys: 42 μs, total: 334 μs
Wall time: 275 μs


In [6]:
df = pd.read_csv(inFile)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 14265041 entries, 0 to 14265040
Data columns (total 51 columns):
 #   Column          Dtype  
---  ------          -----  
 0   DateTime        str    
 1   Open            float64
 2   High            float64
 3   Low             float64
 4   Close           float64
 5   haBody          float64
 6   haWickTop       float64
 7   haWickBott      float64
 8   haColour        int64  
 9   jma             float64
 10  jmaD1           float64
 11  jmaD2           float64
 12  jmaSign         int64  
 13  cfb             float64
 14  cfbD1           float64
 15  cfbD2           float64
 16  cfbSign         int64  
 17  rsxJma          float64
 18  rsxJmaD1        float64
 19  rsxJmaD2        float64
 20  rsxJmaSign      int64  
 21  rsxLast         float64
 22  rsxLastD1       float64
 23  rsxLastD2       float64
 24  rsxLastSign     int64  
 25  momRsxJma       float64
 26  momRsxJmaD1     float64
 27  momRsxJmaD2     float64
 28  momRsxJmaSign   int64

In [7]:
df.rename(columns={'DateTime': 'timestamp'}, inplace=True)

## !!!!!!!!!!!!!!!! ##
# reducing it as adaptable Vel looks exactly the same as Vel(Jma)
df.drop(columns=['adpVelJma','adpVelJmaD1','adpVelJmaD2','adpVelJmaSign'], inplace=True) 

In [8]:
sign_cols = df.columns[df.columns.str.endswith("Sign")]
df[sign_cols] = df[sign_cols].astype("int8")

# schema
type_spec = {
    'haColour': 'int8',
}

# Apply conversion
df = df.astype(type_spec)

# cast to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 14265041 entries, 0 to 14265040
Data columns (total 47 columns):
 #   Column          Dtype         
---  ------          -----         
 0   timestamp       datetime64[us]
 1   Open            float64       
 2   High            float64       
 3   Low             float64       
 4   Close           float64       
 5   haBody          float64       
 6   haWickTop       float64       
 7   haWickBott      float64       
 8   haColour        int8          
 9   jma             float64       
 10  jmaD1           float64       
 11  jmaD2           float64       
 12  jmaSign         int8          
 13  cfb             float64       
 14  cfbD1           float64       
 15  cfbD2           float64       
 16  cfbSign         int8          
 17  rsxJma          float64       
 18  rsxJmaD1        float64       
 19  rsxJmaD2        float64       
 20  rsxJmaSign      int8          
 21  rsxLast         float64       
 22  rsxLastD1       float64    

In [9]:
if not df["timestamp"].is_monotonic_increasing:
    raise ValueError("Unexpected unordered timestamp rows found")

df['date'] = df['timestamp'].dt.normalize()

day = pd.Timestamp("2025-11-28")
df = df[df["date"] != day]

In [10]:
outFile= f'./{symbol}-ha-{SEC}sec-labeller.pqt'

df.to_parquet(outFile, index=False)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
Index: 14256735 entries, 0 to 14265040
Data columns (total 48 columns):
 #   Column          Dtype         
---  ------          -----         
 0   timestamp       datetime64[us]
 1   Open            float64       
 2   High            float64       
 3   Low             float64       
 4   Close           float64       
 5   haBody          float64       
 6   haWickTop       float64       
 7   haWickBott      float64       
 8   haColour        int8          
 9   jma             float64       
 10  jmaD1           float64       
 11  jmaD2           float64       
 12  jmaSign         int8          
 13  cfb             float64       
 14  cfbD1           float64       
 15  cfbD2           float64       
 16  cfbSign         int8          
 17  rsxJma          float64       
 18  rsxJmaD1        float64       
 19  rsxJmaD2        float64       
 20  rsxJmaSign      int8          
 21  rsxLast         float64       
 22  rsxLastD1       float64       
 

In [11]:

print(df.head(20))


             timestamp          Open          High           Low       Close    haBody  haWickTop  haWickBott  haColour           jma     jmaD1     jmaD2  jmaSign        cfb     cfbD1  ...  dblRsxLast  dblRsxLastD1  dblRsxLastD2  dblRsxLastSign    velJma  velJmaD1  velJmaD2  velJmaSign  dblRsxJma  dblRsxJmaD1  dblRsxJmaD2  dblRsxJmaSign   bestBid   bestAsk       date
0  2022-01-03 09:00:04  16430.373047  16430.750000  16429.000000  16429.7500  0.623047   0.376953    0.750000        -1  16430.476562 -0.097656 -0.119141       -1  12.506445 -0.097254  ...   84.519081     -1.307220     -5.145996              -1  0.060596  0.010528 -0.005545           1  86.863152     6.661697    -3.895248              1  16381.75  16382.00 2022-01-03
1  2022-01-03 09:00:06  16430.062500  16430.062500  16429.500000  16429.6250  0.437500   0.000000    0.125000        -1  16430.318359 -0.158203 -0.060547       -1  12.539017  0.032572  ...   77.046646     -7.472435     -6.165215              -1  0.066324  0.00